# Badini GEC — ByT5 vs RAG Corrector Comparison

Compares two approaches to Badini grammatical error correction on the **same** held-out test set:
1. **ByT5** — a fine-tuned byte-level seq2seq model (task-specific, learned).
2. **RAG corrector** — retrieve similar (wrong→correct) examples + a free open LLM imitates them.

Both are scored with precision / recall / F0.5 / GLEU / exact-match, and we pick the winner.

> Honest note: GEC is a *learned* task, not a retrieval task. We expect ByT5 to win; that finding is itself the result. RAG quality is capped by how well the open LLM already knows Badini.

**Runtime: GPU (T4 is fine).** Runtime → Change runtime type → T4 GPU.

In [ ]:
# 1. Clone your repo (has the corpus + synthetic pairs) and install deps
!git clone https://github.com/YOUR_USERNAME/badini-gec.git
%cd badini-gec
!pip -q install transformers datasets accelerate sentencepiece sentence-transformers faiss-cpu

---
## Part A — Fine-tune ByT5

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

def load_pairs(path):
    rows = [l.rstrip('\n').split('\t') for l in open(path, encoding='utf-8')]
    return [r for r in rows if len(r) >= 2]

train_rows = load_pairs('data/pairs_train.tsv')
test_rows  = load_pairs('data/pairs_test.tsv')
print('train', len(train_rows), 'test', len(test_rows))

MODEL = 'google/byt5-small'   # small = fast on T4; bump to byt5-base if memory allows
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

ds = Dataset.from_dict({'src':[r[0] for r in train_rows], 'tgt':[r[1] for r in train_rows]})
ds = ds.train_test_split(test_size=0.05, seed=13)

def prep(b):
    x = tok(b['src'], max_length=256, truncation=True)
    y = tok(text_target=b['tgt'], max_length=256, truncation=True)
    x['labels'] = y['input_ids']; return x
ds = ds.map(prep, batched=True, remove_columns=['src','tgt'])

args = Seq2SeqTrainingArguments(
    output_dir='byt5-badini', per_device_train_batch_size=8,
    gradient_accumulation_steps=4, learning_rate=3e-4, num_train_epochs=3,
    eval_strategy='epoch', save_strategy='no', logging_steps=50,
    bf16=torch.cuda.is_bf16_supported(), predict_with_generate=True)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds['train'],
    eval_dataset=ds['test'], data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()

In [ ]:
# ByT5 inference function
model.eval()
def byt5_correct(text):
    ids = tok(text, return_tensors='pt', truncation=True, max_length=256).input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_length=256, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

print(byt5_correct(test_rows[0][0]))

---
## Part B — RAG corrector
Retrieve the most similar (wrong→correct) examples from the training pairs, put them in a prompt, and have a free open LLM produce the correction.

In [ ]:
# B1. Build the retrieval index over the TRAINING pairs (the knowledge store)
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

# multilingual embedder (handles Arabic script reasonably for a low-resource lang)
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

train_wrong = [r[0] for r in train_rows]
train_right = [r[1] for r in train_rows]
emb = embedder.encode(train_wrong, convert_to_numpy=True, show_progress_bar=True,
                      normalize_embeddings=True)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print('indexed', index.ntotal, 'examples')

def retrieve(text, k=5):
    q = embedder.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q, k)
    return [(train_wrong[i], train_right[i]) for i in I[0]]

In [ ]:
# B2. Load a free open instruction LLM (Qwen2.5-1.5B-Instruct: small, multilingual, no API key)
from transformers import AutoModelForCausalLM, AutoTokenizer as AutoTokCausal
LLM = 'Qwen/Qwen2.5-1.5B-Instruct'
llm_tok = AutoTokCausal.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(LLM, torch_dtype=torch.bfloat16, device_map='auto')

def rag_correct(text, k=5):
    examples = retrieve(text, k)
    shots = '\n'.join([f'Incorrect: {w}\nCorrect: {r}' for w, r in examples])
    prompt = (
        'You are a Badini Kurdish (Arabic script) spelling and grammar corrector. '
        'Using the examples, output ONLY the corrected sentence, nothing else.\n\n'
        f'{shots}\n\nIncorrect: {text}\nCorrect:'
    )
    msgs = [{'role':'user','content':prompt}]
    inp = llm_tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(llm.device)
    with torch.no_grad():
        out = llm.generate(inp, max_new_tokens=80, do_sample=False)
    resp = llm_tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()
    return resp.split('\n')[0].strip()  # first line only

print(rag_correct(test_rows[0][0]))

---
## Part C — Evaluate BOTH on the same held-out test set

In [ ]:
import difflib
from collections import Counter

def edits(s, t):
    sm = difflib.SequenceMatcher(a=s, b=t, autojunk=False); ops=set()
    for tag,i1,i2,j1,j2 in sm.get_opcodes():
        if tag!='equal': ops.add((i1,i2,tuple(t[j1:j2])))
    return ops

def prf(src, hyp, ref, beta=0.5):
    tp=fp=fn=0
    for s,h,r in zip(src,hyp,ref):
        s,h,r=s.split(),h.split(),r.split()
        he,re_=edits(s,h),edits(s,r)
        tp+=len(he&re_); fp+=len(he-re_); fn+=len(re_-he)
    p=tp/(tp+fp) if tp+fp else 1.0; rc=tp/(tp+fn) if tp+fn else 1.0
    f=((1+beta**2)*p*rc/(beta**2*p+rc)) if p+rc else 0.0
    return p,rc,f

def gleu(hyp, ref, n=4):
    num=den=0
    for h,r in zip(hyp,ref):
        h,r=h.split(),r.split()
        for k in range(1,n+1):
            hg=Counter(tuple(h[i:i+k]) for i in range(len(h)-k+1))
            rg=Counter(tuple(r[i:i+k]) for i in range(len(r)-k+1))
            num+=sum((hg&rg).values()); den+=max(sum(hg.values()),1)
    return num/max(den,1)

def report(name, src, hyp, ref):
    p,rc,f=prf(src,hyp,ref); g=gleu(hyp,ref)
    em=sum(h==r for h,r in zip(hyp,ref))/len(ref)
    print(f'{name:8} | P {p:.3f}  R {rc:.3f}  F0.5 {f:.3f}  GLEU {g:.3f}  EM {em:.3f}')
    return dict(model=name, precision=p, recall=rc, f05=f, gleu=g, em=em)

In [ ]:
# Run both correctors over the test set (RAG is slower — this may take a few minutes)
from tqdm import tqdm
src = [r[0] for r in test_rows]
ref = [r[1] for r in test_rows]

byt5_hyp = [byt5_correct(s) for s in tqdm(src, desc='ByT5')]
rag_hyp  = [rag_correct(s)  for s in tqdm(src, desc='RAG')]

print('\n=== RESULTS (same 233 held-out test pairs) ===')
r1 = report('ByT5', src, byt5_hyp, ref)
r2 = report('RAG',  src, rag_hyp,  ref)

winner = 'ByT5' if r1['f05'] >= r2['f05'] else 'RAG'
print(f'\nWINNER on F0.5: {winner}')

In [ ]:
# Save predictions + a results table for your report
import pandas as pd, json
pd.DataFrame({'source':src,'reference':ref,'byt5':byt5_hyp,'rag':rag_hyp}).to_csv('data/comparison_outputs.csv', index=False)
pd.DataFrame([r1,r2]).to_csv('data/comparison_metrics.csv', index=False)
print(json.dumps([r1,r2], indent=2, ensure_ascii=False))
print('\nSaved: data/comparison_outputs.csv  +  data/comparison_metrics.csv')
# To swap in REAL errors later: replace data/pairs_test.tsv with your benchmark and re-run Parts A(eval)–C.

In [ ]:
# (Optional) save the trained ByT5 to Drive so it survives the session
from google.colab import drive; drive.mount('/content/drive')
trainer.save_model('/content/drive/MyDrive/byt5-badini'); tok.save_pretrained('/content/drive/MyDrive/byt5-badini')
print('saved ByT5 to Drive')